note utili, recuperare i VISCODE e VISCODE2 utili per il confronto
non cancellare single visists o variabili con troppi nan --> principalmente in cleaning 2
perchè potrebbero esserci altri file che riempiono questi buchi

In [ ]:
# Seleziona solo le righe in cui TUTTI i volumi richiesti NON sono NaN
vol_cols = [
    "Ventricles%ICV",
    "Hippocampus%ICV",
    "Entorhinal%ICV",
    "Fusiform%ICV",
    "MidTemp%ICV",
    "ICV%ICV"
]
df_merge_vol = df_merge.dropna(subset=vol_cols, how="any").copy()

# Di che colonne identificative vogliamo il confronto?
# Ragionevolmente, 'RID' (subject id) -- eseguiamo il confronto su questo campo.

# Set di RIDs nei due dataframe
rids_merge_vol = set(df_merge_vol["RID"].unique())
rids_vol = set(df_vol["RID"].unique())

# 1) Quanti soggetti in comune hanno i due df
common_rids = rids_merge_vol & rids_vol
print("Numero di soggetti in comune tra df_merge_vol e df_vol:", len(common_rids))

# 2) Quanti sogg ha df_vol che df_merge_vol non ha
only_in_vol = rids_vol - rids_merge_vol
print("Numero di soggetti che sono in df_vol ma non in df_merge_vol:", len(only_in_vol))


In [ ]:
print('rows ADNIMERGE', len(df_merge))
print('rows df_vol', len(df_vol))
print('rows df_merge_vol', len(df_merge_vol))


In [ ]:
import pandas as pd

# Assicurati che le colonne EXAMDATE siano in formato datetime
df_vol['EXAMDATE'] = pd.to_datetime(df_vol['EXAMDATE'], errors='coerce')
df_merge_vol['EXAMDATE'] = pd.to_datetime(df_merge_vol['EXAMDATE'], errors='coerce')

# Per velocità, possiamo ordinare i dataframe
df_merge_vol_sorted = df_merge_vol.sort_values(['RID', 'EXAMDATE'])
df_vol_sorted = df_vol.sort_values(['RID', 'EXAMDATE'])

common_count = 0
only_in_vol_count = 0

idxs_in_common = []
idxs_only_in_vol = []

# Per ciascuna riga di df_vol, verifichiamo se esiste una riga dello stesso RID in df_merge_vol con EXAMDATE a +/- 15gg
for idx, row in df_vol_sorted.iterrows():
    rid = row['RID']
    examdate = row['EXAMDATE']
    sub_merge = df_merge_vol_sorted[df_merge_vol_sorted['RID'] == rid]
    # Trova se c'è almeno un examdate entro +/-15 giorni
    mask_in_range = sub_merge[
        (sub_merge['EXAMDATE'] - examdate).abs().dt.days <= 80
    ]
    if not mask_in_range.empty:
        common_count += 1
        idxs_in_common.append(idx)
    else:
        only_in_vol_count += 1
        idxs_only_in_vol.append(idx)

print(f"Numero di righe in comune tra df_vol e df_merge_vol (stesso RID e EXAMDATE a +/-15gg): {common_count}")
print(f"Numero di righe di df_vol che NON hanno match in df_merge_vol con stesso RID ed EXAMDATE a +/-15gg: {only_in_vol_count}")




In [ ]:
# Seleziona le righe di df_vol che NON hanno un match in df_merge_vol con stesso RID ed EXAMDATE a +/-15gg
df_only_in_vol = df_vol_sorted.loc[idxs_only_in_vol]
print("\nPrime 10 righe di df_vol che df_merge_vol non ha (confronto su RID + EXAMDATE +/-80gg):")
display(df_only_in_vol.head(10))

In [ ]:
import numpy as np
# Calcola quanti soggetti (RID) in df_only_in_vol sono anche presenti in df_merge_vol (indipendentemente dalla data)
unique_rid_only_in_vol = df_only_in_vol['RID'].unique()
unique_rid_merge_vol = df_merge_vol['RID'].unique()
unique_rid_merge = df_merge['RID'].unique()

count_rid_in_both = sum(np.isin(unique_rid_only_in_vol, unique_rid_merge_vol))
count_rid_in_original = sum(np.isin(unique_rid_only_in_vol, unique_rid_merge))
print(f"Numero di soggetti (RID) in df_only_in_vol che sono presenti anche in df_merge_vol: {count_rid_in_both} / {len(unique_rid_only_in_vol)}")
print(f"Numero di soggetti (RID) in df_only_in_vol che sono presenti anche in ADNIMERGE: {count_rid_in_original} / {len(unique_rid_only_in_vol)}")
# Trova i soggetti (RID) in df_only_in_vol che non sono mai presenti in ADNIMERGE (unique_rid_merge)
rids_not_in_adnimerge = [rid for rid in unique_rid_only_in_vol if rid not in unique_rid_merge]
print(f"Soggetti in df_only_in_vol CHE NON sono in ADNIMERGE (unique_rid_merge): {rids_not_in_adnimerge}")
print(f"Totale: {len(rids_not_in_adnimerge)}")


In [ ]:
df_out = df_vol[df_vol['RID'].isin(rids_not_in_adnimerge)]
len(df_out)


In [ ]:
df_merge_vol[df_merge_vol['RID']==123]